# 06b · GSE65391 · RNA_array · do module scores follow disease activity within a child?

Reads `projected_scores.rds` (notebook 05b). Writes `data/run_artifacts/GSE65391/modules_across_visits.csv`.

**Samples.** All SLE samples: 924 samples from 158 children, the first visits and every later visit,
scored on the first-visit modules (notebook 05b).

**Why a mixed model.** Visits of the same child are not independent, so a correlation over all 924
samples would count each child many times. A linear mixed model with a random intercept for each
child accounts for this (Pinheiro JC, Bates DM. *Mixed-Effects Models in S and S-PLUS*. Springer,
2000; R package `nlme`).

**Within and between.** SLEDAI is split into two parts (Neuhaus JM, Kalbfleisch JD. Between- and
within-cluster covariate effects in the analysis of clustered data. *Biometrics* 1998;54:638–645):
- `sledai_between`: the child's mean SLEDAI over all their visits;
- `sledai_within`: the visit's SLEDAI minus that child's mean.

The within coefficient answers "when this child's disease is more active, does the module score
rise?". The between coefficient answers "do children with more active disease on average have higher
scores?". Children with one visit contribute to the between part only.

**Model, for each module:** `score ~ sledai_within + sledai_between`, random intercept per child,
REML (`nlme::lme`).

**Test.** The within-child coefficient, t and p for each module; BH q across the 16 modules.
**A finding** is q < 0.05.

**Comparison with Banchereau et al. 2016 (claim B3).** The paper reports that "the plasmablast
signature was the most robust biomarker of disease activity" in longitudinal data. If this holds
here, the plasma-cell module (greenyellow) should have one of the strongest within-child
associations.

In [1]:
source("../src/paths.R")
suppressMessages(library(nlme))
p <- readRDS(art("GSE65391", "projected_scores.rds"))
m <- p$meta; S <- p$scores
sle <- rownames(m)[m$disease == "SLE" & !is.na(m$sledai)]
d <- data.frame(subject = m[sle, "subject"], sledai = m[sle, "sledai"], S[sle, ])
d$sledai_between <- ave(d$sledai, d$subject)
d$sledai_within  <- d$sledai - d$sledai_between
c(samples = nrow(d), children = length(unique(d$subject)),
  children_with_2_or_more_visits = sum(table(d$subject) >= 2))

samples                       children 
                           924                            158 
children_with_2_or_more_visits 
                           131

**Result.** 924 SLE samples from 158 children; 131 children have two or more visits.

In [2]:
modules <- colnames(S)
fit <- do.call(rbind, lapply(modules, function(mo) {
  f <- lme(as.formula(paste(mo, "~ sledai_within + sledai_between")), random = ~ 1 | subject, data = d, method = "REML")
  tt <- summary(f)$tTable
  data.frame(module = mo,
             within_coef = tt["sledai_within", "Value"], within_t = tt["sledai_within", "t-value"], within_p = tt["sledai_within", "p-value"],
             between_coef = tt["sledai_between", "Value"], between_t = tt["sledai_between", "t-value"], between_p = tt["sledai_between", "p-value"])
}))
fit$within_q  <- p.adjust(fit$within_p, "BH")
fit$between_q <- p.adjust(fit$between_p, "BH")
fit <- fit[order(-abs(fit$within_t)), ]
format(fit[, c("module", "within_coef", "within_t", "within_q", "between_coef", "between_t", "between_q")], digits = 3)

,module,within_coef,within_t,within_q,between_coef,between_t,between_q
,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>
10,greenyellow,0.003364,8.711,2.96e-16,0.003067,3.465,2.74e-03
2,pink,0.003177,7.296,5.95e-12,0.004188,3.968,5.89e-04
8,purple,-0.002313,-5.154,1.74e-06,-0.001846,-2.025,7.53e-02
14,lightcyan,0.001817,4.912,4.42e-06,0.004792,4.571,7.84e-05
11,black,-0.002061,-4.280,6.75e-05,-0.004819,-4.765,6.87e-05
5,brown,-0.001838,-3.954,2.24e-04,0.002265,2.001,7.53e-02
3,blue,0.001283,2.729,1.48e-02,0.002361,2.413,3.39e-02
16,cyan,0.001009,2.258,4.85e-02,-0.003848,-3.344,3.31e-03
4,red,0.000931,2.090,6.56e-02,-0.002664,-2.513,2.97e-02


**Result.** Within children, 8 of 16 modules change with SLEDAI at q < 0.05. The strongest are
greenyellow (plasma cell, t = 8.7, q = 3 × 10⁻¹⁶), pink (interferon, t = 7.3), purple (monocyte,
falling, t = −5.2), lightcyan (neutrophil granule, t = 4.9), black (falling, t = −4.3) and brown (red
cell, falling, t = −4.0). Between children, the strongest are lightcyan (t = 4.6), black (t = −4.8) and
pink (t = 4.0).

**Comparison with claim B3.** Rank of the plasma-cell module (greenyellow) among the 16 modules by the
size of its within-child t statistic.

In [3]:
c(greenyellow_rank_within = which(fit$module == "greenyellow"), modules = nrow(fit))
fit[fit$module %in% c("greenyellow", "pink", "lightcyan", "blue"), c("module", "within_t", "within_q", "between_t", "between_q")]

greenyellow_rank_within                 modules 
                      1                      16

,module,within_t,within_q,between_t,between_q
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
10,greenyellow,8.710912,2.955261e-16,3.465274,0.0027369263
2,pink,7.295817,5.949839e-12,3.967696,0.0005886742
14,lightcyan,4.911568,4.421045e-06,4.571398,0.0000784206
3,blue,2.729481,1.483241e-02,2.413293,0.0339363999


**Result: claim B3 reproduced.** The plasma-cell module has the strongest within-child association
with disease activity of all 16 modules: when a child's SLEDAI rises, the plasma-cell score rises.
This is the paper's finding that the plasmablast signature is the most robust marker of disease
activity. Cross-sectionally, at first visits only, interferon was slightly stronger (notebook 06); the
longitudinal design reverses that order, as the paper's design would predict.

In [4]:
write.csv(fit, art("GSE65391", "modules_across_visits.csv"), row.names = FALSE)